## 02. Feature Engineering — ESS 배터리 수명 예측 

**목표** 
- DAY 1 피처 구성을 `src/features.py`로 계산, 제외 규칙으로 모델링 셀 선택, **DAY 1 수치 일치**, **누수 없음** 확인. 학습, 평가는 `03_modeling.ipynb`.

**피처 3개**(모두 cycle 100 이하): 
- 핵심 `dQ_log_var` = ΔQ(V) = Q₁₀₀(V) − Q₁₀(V) 곡선 분산의 log10(`src.eda_utils.early_features`) 
- 보조 `avgC_0to80_meas` = cycle 10 실측 전류로 잰 SOC 0→80% 평균 충전 속도(C)(`src.eda_utils.charge_current_table`) 
- 후보 `Tavg_mean_2_100` = cycle 2~100 평균 온도(°C)(`src.eda_utils.early_features`)

In [1]:
import os, sys, copy

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())          

import numpy as np
import pandas as pd

from src.load_data import load_all       # 캐시(data/cache/b*.pkl) 읽기
from src import features as F            # DAY 2 피처 표 함수
from src.features import CORE, AUX, CAND, FEATURES

pd.set_option("display.width", 160)
print("작업 폴더:", os.path.basename(os.getcwd()))
print("피처 3개:", FEATURES, "| 돌출 셀 기준(Ah):", F.PROTRUSION_AH)

작업 폴더: Data_mini
피처 3개: ['dQ_log_var', 'avgC_0to80_meas', 'Tavg_mean_2_100'] | 돌출 셀 기준(Ah): 0.002


### 1. 캐시 load

In [3]:
A = load_all()

n_cells = {}
for b, cells in A.items():
    n_cells[b] = len(cells)
print("배치별 셀 수:", n_cells, "| 합계", sum(n_cells.values()))
print("Qdlin 크기(사이클 x 전압 점):", A["b1"][0]["Qdlin"].shape)

배치별 셀 수: {'b1': 46, 'b2': 47, 'b3': 46} | 합계 139
Qdlin 크기(사이클 x 전압 점): (100, 1000)


### 2. 피처 표 

- 셀 1개 = 1행, 제외 셀도 `excl` 열로 표시만. 
- 타깃 `logL = log10(cycle_life)`. `protrusion` = ΔQ 최대값 > +2mAh '돌출 셀' 표시.

In [5]:
table = F.build_feature_table(A)
print("표 크기:", table.shape)

# 앞 5행만 확인: 식별 정보 + 피처 3개 + 표시 열 + 타깃
cols = ["cell_id", "batch", "policy", "cycle_life", "excl"] + FEATURES + ["protrusion", "logL"]
print(table[cols].head(5).round(4).to_string(index=False))

n_protrusion = table.groupby("batch").protrusion.sum().to_dict()
print("\n배치별 돌출 셀 수:",
      n_protrusion)

표 크기: (139, 27)
cell_id batch         policy  cycle_life  excl  dQ_log_var  avgC_0to80_meas  Tavg_mean_2_100  protrusion   logL
   b1c0    b1 3.6C(80%)-3.6C      1190.0  True     -5.0149           3.5980          31.6037       False 3.0755
   b1c1    b1 3.6C(80%)-3.6C      1179.0  True     -5.0140           3.5979          31.3303       False 3.0715
   b1c2    b1 3.6C(80%)-3.6C      1177.0  True     -4.7370           3.5980          31.4796       False 3.0708
   b1c3    b1     4C(80%)-4C      1226.0  True     -4.4426           3.9975          29.9422       False 3.0885
   b1c4    b1     4C(80%)-4C      1227.0  True     -4.6477           3.9976          31.4489       False 3.0888

배치별 돌출 셀 수: {'b1': 0, 'b2': 5, 'b3': 10}


### 3. 제외 규칙 

DAY 1 규칙, 타깃을 믿을 수 없는 셀 제외: `nan_life` = `cycle_life` NaN, `not_80pct_only` = 데이터 안에서 80%(0.88Ah) 미도달 = **검열(censoring)** 값(실제 수명 ≥ 기록값).

**3-1. 배치별 모델링 셀 수**

In [6]:
clean = F.model_cells(table)             # 제외 이유 표를 출력, 모델링 셀 반환
n_all = table.groupby("batch").size()
n_clean = clean.groupby("batch").size()
counts = pd.DataFrame({"전체": n_all, "모델링 사용": n_clean, "제외": n_all - n_clean})
print(counts.to_string())
assert n_clean.to_dict() == {"b1": 36, "b2": 39, "b3": 44}

모델링 셀 수: {'b1': 36, 'b2': 39, 'b3': 44}
제외 이유 (nan_life = 수명 NaN, not_80pct_only = 수명은 있으나 80% 미도달):
       nan_life  not_80pct_only
batch                          
b1            0              10
b2            8               0
b3            2               0
       전체  모델링 사용  제외
batch                
b1     46      36  10
b2     47      39   8
b3     46      44   2


**3-2. Batch1에서 빠진 10셀**

In [8]:
# Batch1 제외 셀의 기록 수명과, 남은 36셀의 수명 범위를 비교
excluded_b1 = table[(table.batch == "b1") & table.excl]
kept_b1 = clean[clean.batch == "b1"]
print("\nBatch1 제외 10셀의 cycle_life (80% 미도달 = 검열):",
      sorted(excluded_b1.cycle_life.astype(int).tolist()))
print("Batch1 남은 셀 수명 범위:", int(kept_b1.cycle_life.min()), "~", int(kept_b1.cycle_life.max()))


Batch1 제외 10셀의 cycle_life (80% 미도달 = 검열): [879, 891, 897, 902, 906, 1177, 1179, 1190, 1226, 1227]
Batch1 남은 셀 수명 범위: 534 ~ 1074


- 모델링 셀 Batch1 36 / Batch2 39 / Batch3 44 (DAY 1과 같음).
- 제외: Batch1 10셀은 80% 미도달(검열), Batch2 8셀, Batch3 2셀은 cycle_life NaN.
- Batch1 제외 셀이 장수명 쪽이라 학습 수명 범위가 좁아짐 


### 4. Batch1 상관을 DAY 1 값과 대조 

- DAY 1 Batch1(36셀) 피처–log10(수명) Pearson r(−1 ~ +1): `dQ_log_var` −0.844, `avgC_0to80_meas` −0.594, `Tavg_mean_2_100` −0.156. 
- Batch2/3 상관은 안 봄(피처 선택에 테스트 배치 미사용, 개수, NaN만).

In [9]:
F.check_features(clean)                  # NaN 개수, Batch1 요약 통계, Batch1 상관 출력

# DAY 2 코드로 다시 계산한 상관이 DAY 1 보고서 값과 같은지 확인
b1 = clean[clean.batch == "b1"]
r_now = b1[FEATURES].corrwith(b1.logL).round(3)
r_day1 = pd.Series({CORE: -0.844, AUX: -0.594, CAND: -0.156})
corr_check = pd.DataFrame({"DAY 2 재계산": r_now, "DAY 1 보고서": r_day1, "일치": r_now == r_day1})
print("\n", corr_check.to_string())
assert corr_check["일치"].all()

피처 NaN 개수 (0이어야 함): 0
Batch1 피처 요약:
       dQ_log_var  avgC_0to80_meas  Tavg_mean_2_100  cycle_life
count     36.0000          36.0000          36.0000     36.0000
mean      -3.7909           4.5442          32.8022    788.4167
std        0.2328           0.3508           0.8005    148.6957
min       -4.1789           3.9957          30.3754    534.0000
25%       -4.0204           4.3353          32.2436    681.0000
50%       -3.7865           4.4948          33.0223    772.5000
75%       -3.6619           4.7942          33.3985    871.5000
max       -3.3503           5.3956          33.8950   1074.0000
Batch1 피처 vs log10(수명) Pearson r: {'dQ_log_var': -0.844, 'avgC_0to80_meas': -0.594, 'Tavg_mean_2_100': -0.156}

                  DAY 2 재계산  DAY 1 보고서    일치
dQ_log_var          -0.844     -0.844  True
avgC_0to80_meas     -0.594     -0.594  True
Tavg_mean_2_100     -0.156     -0.156  True


[describe] 세 상관이 소수 셋째 자리까지 DAY 1과 일치 → 같은 피처 계산. `dQ_log_var` 최강(|r| = 0.844), `Tavg_mean_2_100` 약함(|r| = 0.156).

[to modeling] `avgC_0to80_meas`(보조)·`Tavg_mean_2_100`(후보)은 **Batch1 CV가 좋아질 때만** 추가(03 ablation). 상관이 커도 `dQ_log_var`와 정보가 겹칠 수 있어 채택은 CV로만.

### 5. leakage check


**5-1. (a) NaN 점검**

In [10]:
# 모델링 셀 119개 × 피처 3개
n_nan = int(clean[FEATURES].isna().sum().sum())
print("(a) 모델링 셀 피처 NaN 개수:", n_nan, "/ 셀 수", len(clean))

(a) 모델링 셀 피처 NaN 개수: 0 / 셀 수 119


**5-2. (b)+(c) cycle 100 이후 데이터와 `cycle_life`를 지운 복사본으로 다시 계산**

In [12]:
A_cut = copy.deepcopy(A)                 # 원본 A는 그대로 두고 복사본만 자름
for cells in A_cut.values():
    for c in cells:
        for key in c["summary"]:
            c["summary"][key] = c["summary"][key][:100]   # cycle 1~100 만 남김
        c["cycle_life"] = np.nan                           # 타깃 정보 제거
        c["n_cycles"] = 100
table_cut = F.build_feature_table(A_cut)

# 원본 표와 자른 표를 cell_id 순서로 맞춰 비교
check_cols = FEATURES + ["protrusion"]
feat_full = table.set_index("cell_id")[check_cols].sort_index()
feat_cut = table_cut.set_index("cell_id")[check_cols].sort_index()

same = (np.allclose(feat_full[FEATURES], feat_cut[FEATURES], equal_nan=True)
        and (feat_full.protrusion == feat_cut.protrusion).all())
max_diff = float(np.nanmax(np.abs(feat_full[FEATURES].to_numpy() - feat_cut[FEATURES].to_numpy())))
print("(b)+(c) cycle ≤ 100 + cycle_life 제거 후에도 피처 동일한가:", same, "| 최대 절대 차이:", max_diff)
assert same

max_len = max(len(c["summary"]["QD"]) for cells in A_cut.values() for c in cells)
print("자른 복사본의 summary 길이(최대):", max_len)

(b)+(c) cycle ≤ 100 + cycle_life 제거 후에도 피처 동일한가: True | 최대 절대 차이: 0.0
자른 복사본의 summary 길이(최대): 100


- (a) 모델링 셀 119개 피처 NaN 0개. (b)(c) 잘라내고 `cycle_life`를 지운 복사본에서도 피처 3개, 돌출 표시 **완전히 같음**(최대 차이 0) → cycle 100 이후, 수명 값 안 읽음.

- 피처 표는 누수 없음. 

### 6. 정리

- **① 탐색 목적**: DAY 1 피처를 재현 가능한 코드로 만들고 누수가 없는지 확인
- **② 해결 방법**: `src/features.py`(DAY 1 `eda_utils` 재사용), 제외 규칙, 잘라낸 복사본으로 재계산
- **③ 전/후 차이**: 139셀 → 모델링 119셀(36/39/44), Batch1 상관 −0.844/−0.594/−0.156 = DAY 1과 같음
- **④ 시사점**: 핵심 피처 1개가 강함. 추가 피처는 Batch1 CV로만 판단. 좁은 학습 범위(534~1,074)는 오류 분석에서 다룸
